<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c39-header.png" alt="Nextia Learning · Entity Resolution and Record Matching: Is This the Same Thing?" width="100%">

# Parse into parts

**[Parse into parts](https://learning.nextia-ai.com/courses/record-matching/m02/parse-into-parts/)** · Entity Resolution and Record Matching: Is This the Same Thing? · Module 2, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** split names and addresses into fields with regular expressions, count the records that do not parse, and convert lengths in product titles to one unit.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 3 MB). |
| **You should know** | Normal forms, from [Normalise](https://learning.nextia-ai.com/courses/record-matching/m02/normalise/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (rapidfuzz 3.14.6, jellyfish 1.3.0, datasketch 2.0.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/record-matching/m02/parse-into-parts/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/M02-L02-parse-into-parts/parse-into-parts-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 2 MB) into a folder `larkfield-match` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-match` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Every customer record and every product listing is **synthetic**: our code made them from invented people and products, and then added realistic errors on purpose (typing errors, nicknames, old addresses, shared family emails). Because the code knows the true person behind each record, you can always check an answer. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/data/dataset-card.md) explains the error model.

In [ ]:
# Setup: download the files into larkfield-match/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C39/"
FILES = {
    "data/sources/garden_centres.csv": "2878063d24a38d841f93bb14e07b2d46619fcc4a3909e92ce0238116434d85b2",
    "data/products/feed_a.csv": "b5badaa6f0a813a19bb321263f46950100ef3a4e9a046163ed97e7af60fc94e2",
    "data/products/feed_b.csv": "c6bcc923db6b503859b20b5e4e1833d3376b08bd96b9d65c0392ae07a480e453",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-match").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-match and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-match.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd` and matplotlib. It defines a few small helpers:

- `source(name)` reads one customer source as it is (`"shop"`, `"loyalty"` or `"garden"`); `records()` reads all three in shared columns (`record_id, source, name, email, phone, address, postcode, date_of_birth`), with the values exactly as the sources wrote them.
- `truth()` says which true customer (`entity_id`) each record belongs to, and its split (`train`, `validation` or `test`).
- `pair_measures(pairs)` measures a table of pairs (`record_id_l`, `record_id_r`) that a matcher calls "the same": pair precision, recall and F1 on the test split, as in the project's `match eval`.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, pair measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

FILES_OF = {"shop": "shop_accounts.csv", "loyalty": "loyalty_members.csv", "garden": "garden_centres.csv"}

def source(name):
    """One customer source as it is: its own columns, every value as text."""
    return pd.read_csv(f"data/sources/{FILES_OF[name]}", dtype=str, keep_default_na=False)

def records():
    """All customer records in shared columns, values as the sources wrote them."""
    s, l, g = source("shop"), source("loyalty"), source("garden")
    parts = [
        pd.DataFrame({"record_id": s.record_id, "source": "shop", "name": s.first_name + " " + s.last_name,
                      "email": s.email, "phone": s.phone, "address": s.address + ", " + s.city + " " + s.province,
                      "postcode": s.postcode, "date_of_birth": s.date_of_birth}),
        pd.DataFrame({"record_id": l.record_id, "source": "loyalty", "name": l.full_name, "email": l.email,
                      "phone": l.phone, "address": "", "postcode": l.postcode, "date_of_birth": l.date_of_birth}),
        pd.DataFrame({"record_id": g.record_id, "source": "garden", "name": g.name, "email": g.email,
                      "phone": g.phone, "address": g.address, "postcode": "", "date_of_birth": g.date_of_birth}),
    ]
    return pd.concat(parts, ignore_index=True)

def truth():
    """record_id -> entity_id (the true customer) and split."""
    return pd.read_csv("data/truth.csv", dtype=str)

def ordered(pairs):
    """The same pairs with the smaller record ID first, one row per pair."""
    lo = pairs[["record_id_l", "record_id_r"]].min(axis=1)
    hi = pairs[["record_id_l", "record_id_r"]].max(axis=1)
    out = pd.DataFrame({"record_id_l": lo, "record_id_r": hi})
    return out[out.record_id_l != out.record_id_r].drop_duplicates()

def true_pairs():
    """Every pair of records of the same true customer, with its split."""
    t = truth()
    m = t.merge(t, on="entity_id", suffixes=("_l", "_r"))
    m = m[m.record_id_l < m.record_id_r]
    return m[["record_id_l", "record_id_r", "split_l"]].rename(columns={"split_l": "split"})

RANK = {"train": 0, "validation": 1, "test": 2}

def pair_measures(pairs, split="test"):
    """Pair precision, recall and F1 of the pairs a matcher calls the same (a pair is in the test split when one
    of its records belongs to a test customer)."""
    p = ordered(pairs)
    t = truth().set_index("record_id")
    rank = t.split.map(RANK)
    p = p.assign(split=np.maximum(p.record_id_l.map(rank), p.record_id_r.map(rank)).map({v: k for k, v in RANK.items()}),
                 same=(p.record_id_l.map(t.entity_id) == p.record_id_r.map(t.entity_id)))
    tp_all = true_pairs()
    p, tp_all = p[p.split == split], tp_all[tp_all.split == split]
    tp = int(p.same.sum())
    precision = tp / len(p) if len(p) else 0.0
    recall = tp / len(tp_all) if len(tp_all) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
            "pairs": len(p), "right": tp, "true_pairs": len(tp_all)}

def same_value_pairs(df, col):
    """Every pair of records that share a non-empty value of col."""
    d = df[df[col].notna() & (df[col] != "")][["record_id", col]]
    sizes = d[col].value_counts()
    n_pairs = int((sizes * (sizes - 1) // 2).sum())
    if n_pairs > 20_000_000:   # protect the computer's memory from a key that is too wide
        raise ValueError(f"This key makes {n_pairs:,} pairs: too many to hold in memory. Use a narrower key.")
    m = d.merge(d, on=col, suffixes=("_l", "_r"))
    return m[m.record_id_l < m.record_id_r][["record_id_l", "record_id_r"]].drop_duplicates()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One field, several facts

The garden centres write the name as `SURNAME, FIRST` and the whole address in one field. Run the next cell and look at the first records.

In [ ]:
garden = source("garden")
print(garden[["record_id", "name", "address"]].head(4).to_string(index=False))

## 2. Split the names

With a comma, the part before it is the surname. Without one, the last word is the surname. Each part is then normalised as in the last lesson. Run the next cell: it splits every garden-centre name and shows the names that do not split.

In [ ]:
import re, unicodedata

TITLES = {"mr", "mrs", "ms", "miss", "dr", "m", "mme", "mlle"}

def norm_name(s):
    """Lower case, no accents, punctuation to spaces, no titles (the last lesson's rules)."""
    s = "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))
    s = re.sub(r"[^\w\s]", " ", s.lower().replace("'", ""))
    return " ".join(w for w in s.split() if w not in TITLES)

def split_name(raw):
    """'SURNAME, FIRST' -> (first, last). Without a comma, the last word is the surname."""
    raw = str(raw or "")
    if "," in raw:
        last, first = raw.split(",", 1)
    else:
        words = raw.split()
        first, last = " ".join(words[:-1]), (words[-1] if words else "")
    return norm_name(first), norm_name(last)

garden[["first", "last"]] = garden.name.map(split_name).tolist()
failed = garden[(garden["first"] == "") | (garden["last"] == "")]
print(f"{len(garden):,} garden-centre names; {len(failed)} do not split into a first name and a surname")
print(failed[["record_id", "name", "first", "last"]].head(5).to_string(index=False))

> **Check.** 46 of the 11943 names do not split. Look at them: every one has `M.` in it. `M.` is the French title "Monsieur", and also an initial. The title rule removes it, so a part is left empty. The lesson explains what a matcher does with such records.

## 3. Take the postcode out of the address

In a Canadian address, the postcode is at the end and has a fixed shape: letter, digit, letter, digit, letter, digit, with an optional space or hyphen in the middle. A regular expression finds it.

> **Predict.** In how many garden-centre addresses will the pattern find a postcode?

In [ ]:
POSTCODE_END = re.compile(r"([A-Za-z]\d[A-Za-z])[\s-]?(\d[A-Za-z]\d)\s*$")

def postcode_of(address):
    """The postcode at the end of an address, as six capitals; '' when there is none."""
    m = POSTCODE_END.search(str(address or ""))
    return (m.group(1) + m.group(2)).upper() if m else ""

garden["postcode"] = garden.address.map(postcode_of)
found = (garden.postcode != "").mean()
print(f"postcode found in {found:.1%} of the garden-centre addresses")
print(garden[["address", "postcode"]].head(3).to_string(index=False))

> **Check.** The postcode is found in 100.0% of the addresses. A rule such as "same name and same postcode" can now use every garden-centre record.

## 4. Your turn: lengths in product titles

Feed A writes `4 ft`, feed B writes `1.2 m` for the same product. Write `length_m(title)`: find the first length with the pattern `LENGTH`, and convert it to metres with `TO_METRES`. Then the cell counts the titles of each feed that have a length.

In [ ]:
TO_METRES = {"ft": 0.3048, "in": 0.0254, "cm": 0.01, "m": 1.0}
LENGTH = re.compile(r"(\d+(?:\.\d+)?)\s*(ft|in|cm|m)\b", re.IGNORECASE)

def length_m(title):
    # 1. m = LENGTH.search(str(title)); if there is no match, return None
    # 2. the number is m.group(1), the unit is m.group(2) (use .lower())
    # 3. return the number times TO_METRES[unit], rounded to 2 decimals
    ...

feed_a = pd.read_csv("data/products/feed_a.csv", dtype=str, keep_default_na=False)
feed_b = pd.read_csv("data/products/feed_b.csv", dtype=str, keep_default_na=False)
feed_a["length_m"] = feed_a.title.map(length_m)
feed_b["length_m"] = feed_b.title.map(length_m)
lengths = {"feed_a": int(feed_a.length_m.notna().sum()), "feed_b": int(feed_b.length_m.notna().sum())}
print(lengths)

In [ ]:
expect_hash('lengths', lengths, '360beb4e9d79a1ea')

Now look at one product in the two feeds:

In [ ]:
for pid, feed in [("NF-00001", feed_a), ("CL-01686", feed_b)]:
    row = feed.set_index("product_id").loc[pid]
    print(pid, repr(row.title), "->", row.length_m, "m")

The same product gives 1.22 m and 1.2 m: 4 ft is 1.2192 m, and feed B rounded it. An exact rule says "different". Module 4 compares numbers with a tolerance. The project's parser also finds weights, volumes and packs, such as `(Pack of 2)`: a pack of 2 is another product.

## 5. In the project

```sh
match parse address "2709 boul. Saint-Joseph app. 27, Saint-Ambroise-des-Monts QC G6Z 3W2"
match parse name "TREMBLAY, MARIE-CLAUDE"
match parse title "Mossway Vinyl Garden Hose, 30 ft, Brown (Pack of 2)"
match normalise S-029535          # one record as written and prepared
```

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Exact and rule-based matching](https://learning.nextia-ai.com/courses/record-matching/m02/exact-and-rule-based-matching/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/record-matching/m02/parse-into-parts/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every customer record and every product listing here is synthetic: our code made them from invented people and products. No real person is in this data.